# Spanning Trees and Shortest Paths

```{contents}
:local:
:depth: 2
```

Facilities planning asks two different questions: which cables connect all offices at minimum total cost, and which routes minimize travel from HQ? Both use weights, but their objectives and graph models differ.

## MST Contracts and Disconnected Graphs

```{index} MST Contracts and Disconnected Graphs
```

Kruskal uses undirected edges with explicit distinct vertices. Parallel edges are allowed; self-loops and unknown endpoints are rejected. Negative edge weights are valid for MSTs. If the graph is disconnected, return a minimum spanning forest and report component count; do not call it a spanning tree. An empty graph returns zero edges/cost/components.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var forest=GreedyGraph.Kruskal(vertices,edges,Console.WriteLine);
Console.WriteLine($"cost={forest.Cost}, edges={forest.Edges.Length}, components={forest.Components}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


Here Remote is isolated, so the result has two components and three accepted edges. In general a forest on V vertices and c components has V−c edges. A nonempty connected result has V−1 edges. With tied weights, multiple optimal forests can exist; compare costs and validity rather than requiring one exact edge set.

## Kruskal Invariants and the Cut Argument

```{index} Kruskal Invariants and the Cut Argument
```

Accepted edges are acyclic. Union-find tracks exactly the components connected by accepted edges; an edge with equal roots would create a cycle. A minimum-cost crossing edge is safe: take the cut separating one current component from the rest. An optimal spanning forest that lacks the chosen edge can exchange a crossing edge on the resulting cycle for it without raising cost.

Within each connected input component, this cut argument shows the accepted forest can be extended to an optimum. Acyclicity alone would not prove minimality. The implementation uses union by size and path halving. Sorting costs O(E log E); disjoint-set work is amortized O((V+E) α(V)) under the standard union-find bound. Include O(V+E) setup/output storage. For parallel-edge graphs, avoid replacing log E with log V without an assumption relating them.

## Dijkstra with Nonnegative Directed Arcs

```{index} Dijkstra with Nonnegative Directed Arcs
```

Dijkstra minimizes path cost from one known source. Directed arcs here specify allowed travel direction; supply both directions when modeling two-way roads. Validate every arc, including unreachable ones, before running: weights must be nonnegative, endpoints known, and loops excluded by this teaching interface. Zero weights are valid. Nullable long distances represent infinity without confusing a valid int.MaxValue path with absence.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"HQ",Console.WriteLine);
foreach(string v in vertices)Console.WriteLine($"HQ->{v}: {(routes.Distance[v]?.ToString()??"unreachable")}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


Relaxation replaces a tentative distance only on strict improvement. Each finite tentative distance corresponds to a discovered source route. A fresh minimum-priority pop has the final shortest distance: any hypothetical shorter route would first leave the already settled set with a prefix no longer than that route; nonnegative remaining weights cannot make a later prefix cheaper. Zero weights preserve the argument. Negative edges invalidate this reasoning, even if a particular example happens to produce correct output.

## Stale Queue Entries and Qualified Costs

```{index} Stale Queue Entries and Qualified Costs
```

Improving a distance enqueues a new entry; the old entry remains in the library queue. Compare its stored priority with the current best distance and skip stale entries. Strict improvement also avoids endless equal-distance updates around zero-weight cycles.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"A","B","C","D","Isolate"};
var arcs=new[]{new GreedyGraph.Edge("A","B",10),new GreedyGraph.Edge("A","C",1),new GreedyGraph.Edge("C","B",1),new GreedyGraph.Edge("B","D",0),new GreedyGraph.Edge("D","B",0)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"A",Console.WriteLine);
Console.WriteLine($"D distance={routes.Distance["D"]}, isolate={(routes.Distance["Isolate"] is null)}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


B first enters the queue at ten, then improves to two. Its old ten-priority entry is skipped. Each reachable adjacency list is scanned on its fresh pop; at most E successful relaxations enqueue entries. This lazy-duplicate implementation can hold O(E) queue entries and has O(V+E log(E+1)) time for nontrivial graphs, plus O(V+E) storage. An indexed decrease-key implementation has different queue/storage bounds. Equal priorities need not have stable library ordering; distances are correct and the chosen shortest predecessor route may vary on ties.

## Reconstruct Only Reachable Routes

```{index} Reconstruct Only Reachable Routes
```

A predecessor records the previous vertex on a strict improving route. Follow predecessors backward and reverse with a stack. Return an empty path for a known unreachable vertex, a singleton for the source, and reject an unknown target. Distance/predecessor maps are read-only views; do not fabricate a one-vertex route for an unreachable destination.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"HQ");
foreach(string target in new[]{"HQ","Depot","Remote"})
 Console.WriteLine($"{target}: [{string.Join(" -> ",routes.PathTo(target))}]");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


Every returned route must begin at the source, end at its target, use directed arcs, and sum to the reported distance. Route reconstruction costs O(path length). Long accumulation avoids int overflow; checked arithmetic rejects overflow rather than silently wrapping. With int weights and finite simple paths in a practical array-sized graph, long provides ample range; its policy is still explicit.

## The Cheapest Network Is Not a Shortest-Route Tree

```{index} The Cheapest Network Is Not a Shortest-Route Tree
```

On an undirected triangle, A–B=2, B–C=2, A–C=3. The MST has total cost four, but its A-to-C route costs four; the direct road costs three. Thus minimal network total does not imply minimal source routes.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"A","B","C"};
var edges=new[]{new GreedyGraph.Edge("A","B",2),new GreedyGraph.Edge("B","C",2),new GreedyGraph.Edge("A","C",3)};
var forest=GreedyGraph.Kruskal(vertices,edges);
var arcs=edges.Concat(edges.Select(e=>new GreedyGraph.Edge(e.To,e.From,e.Weight))).ToArray();
var routes=GreedyGraph.Dijkstra(vertices,arcs,"A");
Console.WriteLine($"MST total={forest.Cost}, MST A-C route=4, shortest A-C={routes.Distance["C"]}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


Choose the algorithm by the objective and graph assumptions, not keywords in a scenario string. Directed road routing is not an undirected MST problem. Neither model automatically solves a tour visiting every store once or a network with resilience/capacity constraints.

## Exercise

```{index} Exercise
```

Verify a distance larger than int.MaxValue, a zero edge, and an unreachable vertex. Print route and distance; reject a negative arc even in a disconnected component.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"A","B","C","Remote"};
var arcs=new[]{new GreedyGraph.Edge("A","B",int.MaxValue),new GreedyGraph.Edge("B","C",1)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] vertices={"A","B","C","Remote"};
var arcs=new[]{new GreedyGraph.Edge("A","B",int.MaxValue),new GreedyGraph.Edge("B","C",1)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"A");
Console.WriteLine($"C={routes.Distance["C"]}, route={string.Join("->",routes.PathTo("C"))}, remote path length={routes.PathTo("Remote").Length}");
try{GreedyGraph.Dijkstra(vertices,arcs.Append(new GreedyGraph.Edge("Remote","C",-1)).ToArray(),"A");}catch(ArgumentOutOfRangeException){Console.WriteLine("unreachable negative arc rejected");}
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


C=2147483648, route=A->B->C, remote path length=0
unreachable negative arc rejected


```{rubric} Footnotes
```
[^1]: Union-find’s near-constant amortized bound assumes union by size/rank and path compression; the inverse Ackermann factor α grows extremely slowly.
[^2]: Dijkstra requires nonnegative edges. Algorithms for negative edges, including Bellman–Ford, belong in a later course; a validation failure is not a reason to silently run Dijkstra anyway.